In [ ]:
import serial
import time

# 解决了 AttributeError 的问题后，这段代码现在可以正常工作了

# 1. 根据日志分析设置串口参数
PORT = '/dev/ttyUSB0'  # 如果你在 Linux 或 macOS 上，端口名会是类似 '/dev/ttyUSB0' 的格式
BAUDRATE = 9600
# serial.EIGHTBITS 现在可以被正确找到了
BYTESIZE = serial.EIGHTBITS
PARITY = serial.PARITY_NONE
STOPBITS = serial.STOPBITS_ONE
TIMEOUT = 1  # 读取超时时间，单位秒

# 2. 上位机要发送的指令
command_to_send = bytes([0x01, 0x03, 0x00, 0x00, 0x00, 0x0D, 0x84, 0x0F])

ser = None  # 在 try 块外部先定义变量
try:
    # 3. 初始化并打开串口
    ser = serial.Serial(
        port=PORT,
        baudrate=BAUDRATE,
        bytesize=BYTESIZE,
        parity=PARITY,
        stopbits=STOPBITS,
        timeout=TIMEOUT
    )

    if ser.is_open:
        print(f"串口 {PORT} 已成功打开。")

        # 4. 清空缓冲区
        ser.reset_input_buffer()
        ser.reset_output_buffer()
        
        # 5. 发送指令
        print(f"发送指令: {' '.join(f'{b:02X}' for b in command_to_send)}")
        ser.write(command_to_send)

        # 等待一小段时间，让下位机有时间响应
        time.sleep(0.1)

        # 6. 循环读取所有返回的数据
        response_data = ser.read_all() # 使用 read_all() 可以更简洁地读取所有可用数据
        
        if response_data:
            print(f"接收到响应 ({len(response_data)} bytes): {' '.join(f'{b:02X}' for b in response_data)}")
            # 在这里可以添加对 response_data 的解析代码
        else:
            print("在超时时间内未接收到任何数据。")

except serial.SerialException as e:
    # 捕获串口可能不存在或无法打开的异常
    print(f"操作串口时发生错误: {e}")
except Exception as e:
    # 捕获其他可能的异常
    print(f"发生未知错误: {e}")

finally:
    # 7. 确保在程序结束时关闭串口
    if ser and ser.is_open:
        ser.close()
        print(f"串口 {PORT} 已关闭。")

In [ ]:
from robotcontrol import *


logger_init()
logger.info("{0} test beginning...".format(Auboi5Robot.get_local_time()))
Auboi5Robot.initialize()
robot=Auboi5Robot()
handle=robot.create_context()
logger.info(f"robot.rshd={handle}")

try:
    ip='192.168.1.100'
    port=8899
    result=robot.connect(ip,port)
    if result!=RobotErrorType.RobotError_SUCC:
        logger.info(f"connect server{ip}:{port} failed.")
    else:
        robot.robot_startup()
        robot.set_collision_class(6)
        robot.init_profile()
        robot.set_work_mode(1) # 0是仿真模式，会在pad上的窗口内运行
        joint_states=robot.get_joint_status() # 电流，电压，温度 一般没事儿
        robot.set_joint_maxacc((0.5, 0.5, 0.5, 0.5, 0.5, 0.5))
        robot.set_joint_maxvelc((0.1, 0.1, 0.1, 0.1, 0.1, 0.1))
        logger.info(robot.get_current_waypoint())
        joint_radian =(-1.2038687467575073, -0.5232918858528137, 1.565064787864685, 1.3268247842788696, -0.0351983979344368, 0.028632627800107002)
        # joint_radian=(-0.02333599515259266, -0.032502394169569016, -0.22847945988178253, 0.08580251783132553, -0.19762969017028809, 0.09175939112901688)
        # robot.move_joint(joint_radian)
        # joint_radian=(0,0,0,0,0,0)
        # robot.move_joint(joint_radian)
        # joint_radian=(-0.02333599515259266, -0.032502394169569016, -0.22847945988178253, 0.08580251783132553, -0.19762969017028809, 0.09175939112901688)
        # robot.move_joint(joint_radian)
        
        
except RobotError as e:
    logger.error("{0} robot Event:{1}".format(robot.get_local_time(), e))
finally:
    if robot.connected:
        robot.robot_shutdown()
        robot.disconnect()
    Auboi5Robot.uninitialize()
    logger.info(f"{Auboi5Robot.get_local_time()} test completed")

In [ ]:
from robotcontrol import *

logger_init()
logger.info("{0} test beginning...".format(Auboi5Robot.get_local_time()))
Auboi5Robot.initialize()
robot=Auboi5Robot()
handle=robot.create_context()
logger.info(f"robot.rshd={handle}")
ip='192.168.1.100'
port=8899
result=robot.connect(ip,port)
robot.robot_startup()
robot.set_collision_class(6)
robot.init_profile()
robot.set_work_mode(1) # 0是仿真模式，会在pad上的窗口内运行
joint_states=robot.get_joint_status() # 电流，电压，温度 一般没事儿
robot.set_joint_maxacc((0.5, 0.5, 0.5, 0.5, 0.5, 0.5))
robot.set_joint_maxvelc((0.1, 0.1, 0.1, 0.1, 0.1, 0.1))
logger.info(robot.get_current_waypoint())
primary_waypoint=robot.get_current_waypoint()
print(f"当前位置：{primary_waypoint['pos']}，当前姿态：{primary_waypoint['ori']}")
print(type(primary_waypoint['ori'])) # 默认是list 需要转 np.array

In [ ]:
from transforms3d.quaternions import *
from transforms3d.euler import *
try:
    first_waypoint=np.array(primary_waypoint['pos'])+np.array([0,0,0.005])
    print(tuple(first_waypoint))
    print(tuple(primary_waypoint['ori']))
    robot.move_to_target_in_cartesian(tuple(first_waypoint), tuple(primary_waypoint['ori']))
    primary_waypoint['pos']=first_waypoint
except RobotError as e:
    logger.error("{0} robot Event:{1}".format(robot.get_local_time(), e))

In [1]:
# 初始化
from transforms3d.quaternions import *
from transforms3d.euler import *
from robotcontrol import *
from my_aubo_control import MyAuboi10
MyAuboi10.initialize()
robot=MyAuboi10()
if robot.set_and_startup():
        logger.info("Robot setup complete and connected.")
primary_waypoint=robot.get_current_waypoint()
primary_ori_in_quat=quat2euler(np.array(primary_waypoint['ori']))
print(f"当前位置：{primary_waypoint['pos']}，当前姿态：{primary_ori_in_quat}")
# joint_radian=tuple(MyAuboi10._prepare_pos['joint'])
# robot.move_joint(joint_radian)# 移到初始位置

2025-10-02 19:53:45,876 [134450985793344] INFO: Oct 02 2025 19:53:45 test beginning...
2025-10-02 19:53:45,878 [134450985793344] INFO: robot.rshd=0
2025-10-02 19:53:45,880 [134450985793344] INFO: ip=192.168.1.100, port=8899
2025-10-02 19:53:58,264 [134450985793344] INFO: {'joint': [-0.8447615504264832, -1.1192034482955933, 0.30673453211784363, -1.3298031091690063, 0.5312808752059937, -0.39653947949409485], 'ori': [0.41053433714381493, -0.461914151137129, -0.6180384194491941, 0.4859273475427126], 'pos': [0.5661683419889317, -1.0629924008045426, 0.4193466456696265]}
2025-10-02 19:53:58,453 [134450985793344] INFO: 正在等待ROS服务 'trigger_calibration_read'...
2025-10-02 19:53:58,466 [134450985793344] INFO: 成功连接到标定服务！


当前位置：[0.5661683419889317, -1.0629924008045426, 0.4193466456696265] <class 'list'> ，当前姿态：[0.41053433714381493, -0.461914151137129, -0.6180384194491941, 0.4859273475427126] <class 'list'>


2025-10-02 19:53:58,468 [134450985793344] INFO: Robot setup complete and connected.


当前位置：[0.5661683419889317, -1.0629924008045426, 0.4193466456696265]，当前姿态：(-1.762977003758553, -0.05857203381087634, 1.8096607895090655)


In [ ]:
try:
    
    tool_pos_on_end=(np.sqrt(0.062), -np.sqrt(0.062), 0.026)
    # tool_pos_on_end=(0,0,0.1)
    tool_ori_on_end=(1, 0, 0, 0)
    tool_desc = {"pos": tool_pos_on_end, "ori": tool_ori_on_end}
    tool_pos_on_base = robot.base_to_base_additional_tool(primary_waypoint['pos'],
                                                          primary_waypoint['ori'],
                                                          tool_desc)
    logger.info(f"tool_pos_on_base={tool_pos_on_base['pos']}")
    rotate_axis = np.array(tool_pos_on_base['pos'])- np.array(primary_waypoint['pos']) # 工具末端相对于法兰盘的位置-法兰盘在基坐标系下的位置=工具末端相对于基坐标系的位置
    # rotate_axis = - np.array(primary_waypoint['pos'])
    logger.info(f"rotate_axis={rotate_axis}")
    # 坐标系默认使用基座坐标系（默认填写下面的值就可以了）
    user_coord = {'coord_type': RobotCoordType.Robot_Base_Coordinate,
                  'calibrate_method': 0,
                  'calibrate_points':
                  {"point1": (0.0, 0.0, 0.0, 0.0, 0.0, 0.0),
                  "point2": (0.0, 0.0, 0.0, 0.0, 0.0, 0.0),
                  "point3": (0.0, 0.0, 0.0, 0.0, 0.0, 0.0)},
                  'tool_desc':
                       {"pos": (0.0, 0.0, 0.0),
                        "ori": (1.0, 0.0, 0.0, 0.0)}
                }
    result=robot.move_rotate(user_coord, tuple(rotate_axis), np.deg2rad(0))
    if result == RobotErrorType.RobotError_SUCC:
        print("旋转成功")
        current_waypoint=robot.get_current_waypoint()
        ori_in_quat=quat2euler(np.array(current_waypoint['ori']))
        print(f"当前位置：{current_waypoint['pos']}，当前姿态：{ori_in_quat}")
except RobotError as e:
    logger.error("{0} robot Event:{1}".format(robot.get_local_time(), e))

In [ ]:
try:
    current_waypoint=robot.get_current_waypoint()
    ori_in_quat=quat2euler(np.array(current_waypoint['ori']))
    print(f"当前位置：{current_waypoint['pos']}，当前姿态：{ori_in_quat}")
    ori = euler2quat(np.deg2rad(270), np.deg2rad(-45), np.deg2rad(90))
    robot.move_to_target_in_cartesian(tuple(MyAuboi10._prepare_waypoint['pos']),tuple(ori))
    current_waypoint=robot.get_current_waypoint()
    print(current_waypoint)
    ori_in_quat=quat2euler(np.array(current_waypoint['ori']))
    print(f"当前位置：{current_waypoint['pos']}，当前姿态：{ori_in_quat}")
except RobotError as e:
    logger.error("{0} robot Event:{1}".format(robot.get_local_time(), e))

# 找到初始姿态了，下面开始搞传感器坐标

In [13]:
try:
    robot.move_to_target_in_cartesian(MyAuboi10._prepare_waypoint['pos'],MyAuboi10._prepare_waypoint['ori'])# 先移动到一个上方。目前0力
    robot.move_z_in_step(0.01,'up') # 实测最小距离：0.0001m=0.1mm
    robot.move_x_in_step(0.001,'left')
    robot.move_y_in_step(0.001,'front')
except RobotError as e:
    logger.error("{0} robot Event:{1}".format(robot.get_local_time(), e))

In [ ]:
# 读取数据
result=robot.get_force()
if isinstance(result, float):
    output_str = f"✅ 当前主读数: {result:<15.6f}"
else:
    output_str = f"❌ 读取失败: { result:<30}"
print(output_str, end='\r')
time.sleep(1)

In [ ]:
current_waypoint=robot.get_current_waypoint()
print(current_waypoint)#
# pos': [0.022161103126798365, -1.1122443441054815, 0.4466148733003072] y坐标对齐了传感器上边界，应该还得加0.1mm，因为有余量来着，
# z坐标是正好碰到的意思，再加3mm是传感器底面了
robot.move_z_in_step(0.001,'up')
robot.move_y_in_step(0.00325,'back')
robot.move_y_in_step(0.0001,'back')

In [15]:
# robot.move_z_in_step(0.00250,'down')
robot.move_z_in_step(0.05,'up') # 以这个步进去采集压力比较合适
robot.move_y_in_step(0.05,'front')


RobotError: RobotError type2004 code=10023 msg=move error

In [ ]:
while robot.move_x_in_step(0.1,'left')==RobotErrorType.RobotError_SUCC:
    robot.move_x_in_step(0.1,'left')


KeyboardInterrupt: 